In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
# Configure the base Qwen generation evaluation, retrieval depth, and selected questions.
import json
import subprocess
import sys
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

# Files
EVALUATION_FILE = "data/enron_evaluation_250.csv"
RETRIEVAL_NOTEBOOK = "notebooks/03_retrieval/Retrieval-Final.ipynb"

# Generation model
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Retrieval / generation settings
RETRIEVAL_K = 10
MAX_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 256

# Set to None for all questions.
QUESTION_NUMBERS_TO_RUN = [
    # Likely correct: 30
    12, 20, 24, 25, 30, 36, 38, 43, 44, 49,
    50, 60, 68, 69, 74, 79, 80, 91, 94, 98,
    100, 104, 113, 122, 130, 142, 157, 166, 190, 224,

    # Not answered: 17
    1, 8, 14, 18, 32, 45, 55, 64, 82,
    95, 110, 121, 144, 169, 188, 219, 237,

    # Answered but clearly incorrect: 3
    4, 41, 102
]

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Model: {MODEL_NAME}")
print(f"Top-k retrieval: {RETRIEVAL_K}")

In [ ]:
# Load the evaluation set and select the questions included in the generation experiment.
questions = pd.read_csv(EVALUATION_FILE).reset_index(drop=True)
questions.insert(0, "question_number", range(1, len(questions) + 1))

if QUESTION_NUMBERS_TO_RUN is not None:
    questions = (
        questions
        .set_index("question_number")
        .loc[QUESTION_NUMBERS_TO_RUN]
        .reset_index()
    )

print(f"Questions loaded: {len(questions)}")

In [ ]:
# Run final retrieval for every evaluation question in a separate process.
# Save the retrieved evidence so generation can reuse it without reloading the retrieval models.
RETRIEVAL_RESULTS_FILE = Path("generation_retrieval.json")

questions_to_retrieve = questions[
    ["question_number", "question"]
].to_dict("records")

retrieval_code = f"""
import json

RUN_RETRIEVAL_EXAMPLE = False

get_ipython().run_line_magic(
    'run',
    '-i "notebooks/03_retrieval/Retrieval-Final.ipynb"'
)

questions = {questions_to_retrieve!r}
results = []

for row in questions:
    chunks = retrieve_chunks(
        row["question"],
        k={RETRIEVAL_K}
    )

    results.append(chunks)

    print(
        f"Retrieved question {{row['question_number']}}",
        flush=True
    )

with open(r"{RETRIEVAL_RESULTS_FILE}", "w") as f:
    json.dump(results, f)
"""

process = subprocess.run(
    [sys.executable, "-m", "IPython", "-c", retrieval_code],
    capture_output=True,
    text=True,
)

print(process.stdout)

if process.returncode != 0:
    print(process.stderr)
    raise RuntimeError("Retrieval failed.")

with open(RETRIEVAL_RESULTS_FILE) as f:
    retrieval_results = json.load(f)

print(
    f"Retrieved evidence for "
    f"{len(retrieval_results)} question(s)."
)

print(
    f"Chunks for first question: "
    f"{len(retrieval_results[0])}"
)

In [ ]:
# Inspect the retrieved evidence for the first evaluation question before generation.
for i, chunk in enumerate(retrieval_results[0], start=1):
    print(f"\n--- RESULT {i} ---")
    print("Email ID:", chunk["email_id"])
    print("Subject:", chunk["subject"])
    print("Date:", chunk["date"])
    print("Chunk:")
    print(chunk["chunk_text"][:1000])

In [ ]:
# Define the grounding and citation instructions used for the base Qwen evaluation.
UNKNOWN_RESPONSE = "I could not find that information in the provided emails."

SYSTEM_PROMPT = f"""
Use the supplied email evidence to answer the question.
You may combine information across multiple pieces of evidence.

Do not use outside knowledge or invent information.

If the answer cannot be determined from the supplied evidence, say:
I could not find that information in the provided emails.

Return:
ANSWER: <answer>
SOURCE: <email_id>
""".strip()

# Format the question and retrieved email evidence into the final generation prompt.
def build_prompt(question, chunks):
    evidence_blocks = []

    for chunk in chunks:
        evidence_blocks.append(
            f"EMAIL_ID: {chunk['email_id']}\n"
            f"SUBJECT: {chunk.get('subject', '')}\n"
            f"DATE: {chunk.get('date', '')}\n"
            f"EVIDENCE:\n"
            f"{chunk['chunk_text']}"
        )

    evidence = "\n\n---\n\n".join(evidence_blocks)

    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"QUESTION:\n{question}\n\n"
        f"EMAIL EVIDENCE:\n{evidence}"
    )

In [ ]:
# Inspect the first complete prompt to verify the evidence and instructions sent to Qwen.
test_prompt = build_prompt(
    questions.iloc[0]["question"],
    retrieval_results[0]
)

print(test_prompt[:4000])

In [ ]:
# Load the base Qwen model and tokenizer for generation evaluation.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"": DEVICE},
    attn_implementation="eager",
).eval()

print("Qwen loaded successfully.")

In [ ]:
# Run one evaluation question through base Qwen to verify generation before the full experiment.
question = questions.iloc[0]["question"]
chunks = retrieval_results[0]

prompt = build_prompt(question, chunks)

messages = [
    {"role": "user", "content": prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_INPUT_TOKENS,
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_tokens = output[
    0,
    inputs["input_ids"].shape[1]:
]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(questions.iloc[0]["answer"])

print("\nMODEL RESPONSE:")
print(response)

In [ ]:
# Generate a grounded response for every selected question and record its retrieval context.
# Save results after each question so completed generations survive an interrupted run.
RESULTS_FILE = "results/generation_results_base_qwen.csv"

results = []

for i, (row, chunks) in enumerate(
    zip(questions.itertuples(index=False), retrieval_results),
    start=1
):
    prompt = build_prompt(row.question, chunks)

    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    retrieved_ids = [
        str(chunk["email_id"])
        for chunk in chunks
    ]

    gold_email_id = str(row.email_id)

    if gold_email_id in retrieved_ids:
        gold_email_rank = retrieved_ids.index(gold_email_id) + 1
    else:
        gold_email_rank = None

    results.append({
        "question_number": row.question_number,
        "question": row.question,
        "expected_answer": row.answer,
        "expected_email_id": gold_email_id,
        "difficulty": row.difficulty,
        "expected_email_retrieved": gold_email_id in retrieved_ids,
        "gold_email_rank": gold_email_rank,
        "model_response": response,
        "retrieved_email_ids": " | ".join(retrieved_ids),
    })

    # Save after every question
    pd.DataFrame(results).to_csv(
        RESULTS_FILE,
        index=False
    )

    print(
        f"{i}/{len(questions)} completed - "
        f"Question {row.question_number}"
    )

In [ ]:
# Reload the saved base Qwen responses and verify that the generation run completed.
generation_results = pd.read_csv(
    "results/generation_results_base_qwen.csv"
)

print(f"Saved responses: {len(generation_results)}")

display(generation_results.head())

In [ ]:
# Inspect each generated response alongside its expected answer and retrieval outcome.
for _, row in generation_results.iterrows():
    print("=" * 80)
    print(f"QUESTION {row['question_number']}")

    print("\nQuestion:")
    print(row["question"])

    print("\nExpected answer:")
    print(row["expected_answer"])

    print("\nGold email retrieved:")
    print(row["expected_email_retrieved"])

    print("\nGold email rank:")
    print(row["gold_email_rank"])

    print("\nQwen response:")
    print(row["model_response"])

    print("\nRetrieved email IDs:")
    print(row["retrieved_email_ids"])

    print()

## Binary answer accuracy

An AI judge assigns 1 when the generated answer communicates every fact requested by the question and 0 when it is incorrect, missing, refuses to answer, or omits a required fact. Differences in capitalisation, punctuation, formatting, and equivalent wording do not affect correctness. The conditional result measures generation accuracy only where retrieval returned the labelled gold email.

In [ ]:
# Calculate answer accuracy overall and when the labelled source email was successfully retrieved.
generation_results = pd.read_csv(
    "results/generation_results_base_qwen.csv"
)

overall_correct = int(generation_results["ai_judge_score"].sum())
overall_total = len(generation_results)

gold_retrieved_results = generation_results[
    generation_results["expected_email_retrieved"].astype(bool)
]

retrieved_correct = int(gold_retrieved_results["ai_judge_score"].sum())
retrieved_total = len(gold_retrieved_results)

answer_accuracy = pd.DataFrame({
    "scope": [
        "All 50 questions",
        "Gold email retrieved",
    ],
    "correct_answers": [
        overall_correct,
        retrieved_correct,
    ],
    "questions": [
        overall_total,
        retrieved_total,
    ],
})

answer_accuracy["accuracy"] = (
    answer_accuracy["correct_answers"]
    / answer_accuracy["questions"]
)

display(answer_accuracy.style.format({"accuracy": "{:.1%}"}))


## Citation quality

Citation quality receives 1 when the model returns a correctly formatted source email ID that appears in the retrieved evidence supplied to the model, and 0 when the citation is missing, malformed, or not among the retrieved email IDs. This measures whether the answer links back to an available source without requiring an exact match to the labelled gold email.

In [ ]:
# Calculate the proportion of generated responses that provide a valid source citation.
# Requires the externally assigned citation_quality_score column.
generation_results = pd.read_csv(
    "results/generation_results_base_qwen.csv"
)

citation_correct = int(
    generation_results["citation_quality_score"].sum()
)
citation_total = len(generation_results)
citation_quality = citation_correct / citation_total

citation_summary = pd.DataFrame({
    "model": ["Base Qwen"],
    "valid_citations": [citation_correct],
    "questions": [citation_total],
    "citation_quality": [citation_quality],
})

display(
    citation_summary.style.format({"citation_quality": "{:.1%}"})
)


## AI faithfulness and overall metric summary

AI faithfulness assigns 1 when every factual claim in the generated answer is supported by the cited retrieved email evidence in the context of the question, and 0 when the answer contradicts or misinterprets that evidence. Responses without a valid citation are excluded because there is no usable evidence against which to judge faithfulness. This is deliberately separate from answer accuracy: an answer may faithfully reflect a retrieved email while still disagreeing with the labelled expected answer.


In [ ]:
# Summarise base Qwen answer accuracy, citation quality, and evidence faithfulness.
# Plot the three final generation metrics for comparison with later fine-tuned models.
import matplotlib.pyplot as plt

generation_results = pd.read_csv(
    "results/generation_results_base_qwen.csv"
)

answer_correct = int(generation_results["ai_judge_score"].sum())
answer_total = len(generation_results)
citation_correct = int(generation_results["citation_quality_score"].sum())
citation_total = len(generation_results)
faithfulness_scores = generation_results["ai_faithfulness_score"].dropna()
faithful_correct = int(faithfulness_scores.sum())
faithful_total = len(faithfulness_scores)

answer_accuracy = answer_correct / answer_total
citation_quality = citation_correct / citation_total
ai_faithfulness = faithful_correct / faithful_total

print(f"Answer accuracy: {answer_accuracy:.1%} ({answer_correct}/{answer_total})")
print(f"Citation quality: {citation_quality:.1%} ({citation_correct}/{citation_total})")
print(
    f"AI faithfulness: {ai_faithfulness:.1%} "
    f"({faithful_correct}/{faithful_total} validly cited answers)"
)

metric_summary = pd.DataFrame({
    "metric": [
        "Answer accuracy",
        "Citation quality",
        "AI faithfulness",
    ],
    "score": [
        answer_accuracy,
        citation_quality,
        ai_faithfulness,
    ],
})

fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.bar(
    metric_summary["metric"],
    metric_summary["score"] * 100,
    color=["#4C78A8", "#72B7B2", "#F2CF5B"],
    width=0.62,
)

ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=11)
ax.set_title("Base Qwen evaluation overview")
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 105)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()
